In [1]:
.open earthquakes.db
.load ../../../sqlite-tg/dist/tg0
.load ../../dist/debug/jiff0

[no code]

In [2]:
drop table if exists earthquakes;
create table earthquakes(
  id text primary key,
  place text,
  magnitude real,
  occurred_at timestamp,
  location geometry,
  timezone text
);

insert into earthquakes
select 
  geometry ->> 'id' as id,
  geometry ->> '$.properties.place' as place,
  geometry ->> '$.properties.mag' as magnitude,
  jiff_timestamp_from_ms(geometry ->> '$.properties.time') as occurred_at,
  tg_to_wkt(geometry) as location,
  null as timezone
from tg_each(readfile('eathquakes.geojson'));

select * from earthquakes limit 10;

id,place,magnitude,occurred_at,location,timezone
us6000pfw7,Kuril Islands,6.8,2024-12-27T12:47:37.696Z,POINT(151.236 47.2996 146),
us7000p0lv,"33 km W of Port-Vila, Vanuatu",6.1,2024-12-21T15:30:53.399Z,POINT(168.0033 -17.7041 47),
us7000nzf3,"24 km WNW of Port-Vila, Vanuatu",7.3,2024-12-17T01:47:25.741Z,POINT(168.0842 -17.6914 54.372),
us7000nyqr,"56 km ESE of Molina, Chile",6.4,2024-12-13T23:38:18.234Z,POINT(-70.7256 -35.33 109),
us7000nx5z,"126 km SSW of Adak, Alaska",6.1,2024-12-09T00:38:52.848Z,POINT(-177.443 50.8586 10),
us7000nx5r,"104 km SSW of Adak, Alaska",6.3,2024-12-09T00:15:30.691Z,POINT(-177.1779 50.9985 19),
us7000nx3z,"108 km SSW of Adak, Alaska",6.3,2024-12-08T19:57:08.602Z,POINT(-177.3278 51.0018 18),
us7000nx17,Kuril Islands,6,2024-12-08T10:25:00.081Z,POINT(152.3071 48.867 207),
nc75095651,"2024 Offshore Cape Mendocino, California Earthquake",7,2024-12-05T18:44:21.11Z,POINT(-125.021666666667 40.374 10),
us7000nu90,"38 km WNW of Hakui, Japan",6.1,2024-11-26T13:47:03.411Z,POINT(136.3653 36.9549 8),


In [3]:
drop table if exists tg_timezones;
create virtual table tg_timezones using tg0(tzid);
insert into tg_timezones(rowid, tzid, _shape)
select 
  rowid,
  geometry ->> '$.properties.tzid' as tzid,
  geometry
from tg_each(readfile('timezones.geojson'));
select * from tg_timezones limit 10;

_shape,tzid
Blob<687869>,America/Lima
Blob<1708093>,America/Santiago
Blob<412192>,America/Punta_Arenas
Blob<661837>,America/Coyhaique
Blob<326141>,America/Rio_Branco
Blob<156461>,America/Eirunepe
Blob<690445>,America/Argentina/Rio_Gallegos
Blob<630835>,America/Argentina/Catamarca
Blob<1013155>,America/Argentina/Salta
Blob<664733>,America/Argentina/Mendoza


In [4]:

with earthquake_timezones as (
  select 
    id,
    tg_timezones.tzid
  from earthquakes 
  left join tg_timezones on tg_intersects(_shape, location)
)
update earthquakes
set timezone = earthquake_timezones.tzid
from earthquake_timezones
where earthquakes.id = earthquake_timezones.id;

┌├

In [5]:
select * from earthquakes limit 20;

id,place,magnitude,occurred_at,location,timezone
us6000pfw7,Kuril Islands,6.8,2024-12-27T12:47:37.696Z,POINT(151.236 47.2996 146),Etc/GMT-10
us7000p0lv,"33 km W of Port-Vila, Vanuatu",6.1,2024-12-21T15:30:53.399Z,POINT(168.0033 -17.7041 47),Pacific/Efate
us7000nzf3,"24 km WNW of Port-Vila, Vanuatu",7.3,2024-12-17T01:47:25.741Z,POINT(168.0842 -17.6914 54.372),Pacific/Efate
us7000nyqr,"56 km ESE of Molina, Chile",6.4,2024-12-13T23:38:18.234Z,POINT(-70.7256 -35.33 109),America/Santiago
us7000nx5z,"126 km SSW of Adak, Alaska",6.1,2024-12-09T00:38:52.848Z,POINT(-177.443 50.8586 10),Etc/GMT+12
us7000nx5r,"104 km SSW of Adak, Alaska",6.3,2024-12-09T00:15:30.691Z,POINT(-177.1779 50.9985 19),Etc/GMT+12
us7000nx3z,"108 km SSW of Adak, Alaska",6.3,2024-12-08T19:57:08.602Z,POINT(-177.3278 51.0018 18),Etc/GMT+12
us7000nx17,Kuril Islands,6,2024-12-08T10:25:00.081Z,POINT(152.3071 48.867 207),Etc/GMT-10
nc75095651,"2024 Offshore Cape Mendocino, California Earthquake",7,2024-12-05T18:44:21.11Z,POINT(-125.021666666667 40.374 10),Etc/GMT+8
us7000nu90,"38 km WNW of Hakui, Japan",6.1,2024-11-26T13:47:03.411Z,POINT(136.3653 36.9549 8),Etc/GMT-9


In [6]:
select timezone
from earthquakes
where not jiff_timezone_is_available(timezone)
limit 10;


timezone
America/Coyhaique


In [7]:
select 
  jiff_time_round(
    jiff_time(
      jiff_zoned(occurred_at, timezone)
    ),
    'hour'
  ) as local_time_bucket,
  count(*),
  round(avg(magnitude),2)
from earthquakes
where jiff_timezone_is_available(timezone)
group by 1
order by 1;


local_time_bucket,count(*),"round(avg(magnitude),2)"
00:00:00,96,6.43
01:00:00,89,6.4
02:00:00,78,6.32
03:00:00,75,6.33
04:00:00,96,6.43
05:00:00,90,6.36
06:00:00,76,6.42
07:00:00,93,6.43
08:00:00,76,6.36
09:00:00,84,6.36
